<a href="https://colab.research.google.com/github/AliefReefandi/Grokking-Deep-Learning/blob/main/Chapter_10_Neural_Learning_About_Edges_and_Corners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10 — Neural Learning About Edges and Corners

## Tujuan Pembelajaran

Pada chapter ini kita mempelajari Convolutional Neural Network (CNN)
dan bagaimana CNN dapat digunakan untuk mengenali pola pada gambar.

Setelah menyelesaikan chapter ini, kita diharapkan dapat:

- Memahami konsep reusing weights.
- Memahami alasan penggunaan convolutional layer.
- Memahami konsep kernel/filter.
- Memahami bagaimana kernel mencari pola pada gambar.
- Memahami proses convolution secara sederhana.
- Memahami hubungan CNN dengan overfitting dan generalization.
- Mengimplementasikan convolutional neural network sederhana
  menggunakan NumPy.

## Reusing Weights

Salah satu ide utama dalam chapter ini adalah menggunakan weights
yang sama untuk mendeteksi pola yang sama pada lokasi yang berbeda.

Contohnya, jika sebuah network ingin mendeteksi garis horizontal,
maka network tidak perlu memiliki weights yang berbeda untuk setiap
lokasi gambar.

Weights yang sama dapat digunakan pada banyak lokasi.

Konsep ini disebut sebagai reusing weights.

## Kenapa Reusing Weights Penting?

Bayangkan sebuah gambar berukuran besar.

Jika setiap bagian gambar menggunakan weights yang berbeda,
maka setiap bagian harus belajar sendiri bagaimana mendeteksi
sebuah garis atau pola.

Hal ini menghasilkan banyak parameter dan dapat meningkatkan
kemungkinan overfitting.

Dengan menggunakan weights yang sama, satu kernel dapat belajar
mendeteksi suatu pola dan kemudian mencari pola tersebut di berbagai
lokasi gambar.

Dengan demikian:

Reusing weights
        ↓

Lebih sedikit parameter
        ↓

Lebih sedikit overfitting
        ↓
        
Generalization lebih baik

## Convolutional Layer

Convolutional layer menggunakan sekumpulan weights kecil yang disebut
kernel atau filter.

Kernel digunakan untuk melihat bagian kecil dari sebuah gambar.

Contohnya:

Gambar 28 × 28
       ↓

Kernel 3 × 3
       ↓

Mendeteksi pola kecil
       ↓

Digeser ke lokasi lain
       ↓
       
Menghasilkan feature map

Kernel dapat belajar mendeteksi pola seperti:

- garis horizontal
- garis vertikal
- garis diagonal
- sudut
- pola kecil lainnya

## Kernel

Kernel adalah sekumpulan weights kecil yang digunakan untuk
mendeteksi pola tertentu pada gambar.

Misalnya kernel berukuran 3 × 3:

[ w1  w2  w3 ]
[ w4  w5  w6 ]
[ w7  w8  w9 ]

Kernel tersebut akan dibandingkan dengan bagian kecil dari gambar.

Kemudian nilai-nilai tersebut dikalikan dan dijumlahkan sehingga
menghasilkan sebuah nilai output.

Kernel kemudian digunakan kembali pada bagian gambar berikutnya.

## Bagaimana Convolution Bekerja?

Secara sederhana:

1. Ambil bagian kecil dari gambar.
2. Gunakan kernel pada bagian tersebut.
3. Lakukan perkalian antara input dan weights.
4. Jumlahkan hasil perkalian.
5. Simpan hasil sebagai output.
6. Geser kernel ke lokasi berikutnya.
7. Ulangi sampai seluruh gambar diproses.

Dengan demikian, satu kernel dapat digunakan untuk mencari pola
yang sama di seluruh bagian gambar.

In [1]:
import numpy as np

image = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

kernel = np.array([
    [1, 0],
    [0, 1]
])

result = np.sum(
    image[:2, :2] * kernel
)

print("Hasil convolution:", result)

Hasil convolution: 6


## CNN pada MNIST

Pada implementasi chapter ini, dataset yang digunakan adalah MNIST.

Setiap gambar MNIST berukuran:

28 × 28 pixel

Network menggunakan:

- Kernel 3 × 3
- 16 kernel
- Tanh sebagai activation function
- Softmax pada output
- Dropout
- Batch gradient descent

Kernel digunakan untuk memproses berbagai bagian gambar sebelum
hasilnya diteruskan ke layer berikutnya.

In [2]:
import numpy as np
import sys

np.random.seed(1)

from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (
    x_train[0:1000].reshape(1000, 28 * 28) / 255,
    y_train[0:1000]
)

one_hot_labels = np.zeros((len(labels), 10))

for i, l in enumerate(labels):
    one_hot_labels[i][l] = 1

labels = one_hot_labels

test_images = (
    x_test.reshape(len(x_test), 28 * 28) / 255
)

test_labels = np.zeros((len(y_test), 10))

for i, l in enumerate(y_test):
    test_labels[i][l] = 1

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
def tanh(x):
    return np.tanh(x)


def tanh2deriv(output):
    return 1 - (output ** 2)


def softmax(x):
    temp = np.exp(x)

    return temp / np.sum(
        temp,
        axis=1,
        keepdims=True
    )

In [5]:
alpha = 2
iterations = 300

pixels_per_image = 784
num_labels = 10

batch_size = 128

input_rows = 28
input_cols = 28

kernel_rows = 3
kernel_cols = 3

num_kernels = 16

hidden_size = (
    (input_rows - kernel_rows)
    *
    (input_cols - kernel_cols)
    *
    num_kernels
)

kernels = (
    0.02
    * np.random.random(
        (kernel_rows * kernel_cols, num_kernels)
    )
    - 0.01
)

weights_1_2 = (
    0.2
    * np.random.random(
        (hidden_size, num_labels)
    )
    - 0.1
)

In [6]:
def get_image_section(
    layer,
    row_from,
    row_to,
    col_from,
    col_to
):
    section = layer[
        :,
        row_from:row_to,
        col_from:col_to
    ]

    return section.reshape(
        -1,
        1,
        row_to - row_from,
        col_to - col_from
    )

In [7]:
for j in range(iterations):

    correct_cnt = 0

    for i in range(
        int(len(images) / batch_size)
    ):

        batch_start = i * batch_size
        batch_end = (i + 1) * batch_size

        # Input
        layer_0 = images[
            batch_start:batch_end
        ]

        layer_0 = layer_0.reshape(
            layer_0.shape[0],
            28,
            28
        )

        # Mengambil semua subregion gambar
        sects = []

        for row_start in range(
            layer_0.shape[1] - kernel_rows
        ):

            for col_start in range(
                layer_0.shape[2] - kernel_cols
            ):

                sect = get_image_section(
                    layer_0,
                    row_start,
                    row_start + kernel_rows,
                    col_start,
                    col_start + kernel_cols
                )

                sects.append(sect)

        # Menggabungkan semua subregion
        expanded_input = np.concatenate(
            sects,
            axis=1
        )

        es = expanded_input.shape

        flattened_input = expanded_input.reshape(
            es[0] * es[1],
            -1
        )

        # Convolution
        kernel_output = flattened_input.dot(
            kernels
        )

        # Tanh
        layer_1 = tanh(
            kernel_output.reshape(
                es[0],
                -1
            )
        )

        # Dropout
        dropout_mask = np.random.randint(
            2,
            size=layer_1.shape
        )

        layer_1 *= dropout_mask * 2

        # Output
        layer_2 = softmax(
            np.dot(
                layer_1,
                weights_1_2
            )
        )

        # Accuracy
        for k in range(batch_size):

            labelset = labels[
                batch_start + k:
                batch_start + k + 1
            ]

            correct_cnt += int(
                np.argmax(
                    layer_2[k:k+1]
                )
                ==
                np.argmax(labelset)
            )

        # Backpropagation
        layer_2_delta = (
            labels[batch_start:batch_end]
            - layer_2
        ) / (
            batch_size * layer_2.shape[0]
        )

        layer_1_delta = (
            layer_2_delta.dot(
                weights_1_2.T
            )
            *
            tanh2deriv(layer_1)
        )

        layer_1_delta *= dropout_mask

        # Update output weights
        weights_1_2 += (
            alpha
            *
            layer_1.T.dot(
                layer_2_delta
            )
        )

        # Update kernel
        l1d_reshape = layer_1_delta.reshape(
            kernel_output.shape
        )

        k_update = (
            flattened_input.T.dot(
                l1d_reshape
            )
        )

        kernels -= (
            alpha * k_update
        )

In [8]:
test_correct_cnt = 0

for i in range(len(test_images)):

    layer_0 = test_images[i:i+1]

    layer_0 = layer_0.reshape(
        layer_0.shape[0],
        28,
        28
    )

    sects = []

    for row_start in range(
        layer_0.shape[1] - kernel_rows
    ):

        for col_start in range(
            layer_0.shape[2] - kernel_cols
        ):

            sect = get_image_section(
                layer_0,
                row_start,
                row_start + kernel_rows,
                col_start,
                col_start + kernel_cols
            )

            sects.append(sect)

    expanded_input = np.concatenate(
        sects,
        axis=1
    )

    es = expanded_input.shape

    flattened_input = expanded_input.reshape(
        es[0] * es[1],
        -1
    )

    kernel_output = flattened_input.dot(
        kernels
    )

    layer_1 = tanh(
        kernel_output.reshape(
            es[0],
            -1
        )
    )

    layer_2 = np.dot(
        layer_1,
        weights_1_2
    )

    test_correct_cnt += int(
        np.argmax(layer_2)
        ==
        np.argmax(
            test_labels[i:i+1]
        )
    )

In [9]:
print(
    "Test Accuracy:",
    test_correct_cnt / float(len(test_images))
)

print(
    "Train Accuracy:",
    correct_cnt / float(len(images))
)

Test Accuracy: 0.8749
Train Accuracy: 0.824


## Kenapa CNN Lebih Baik untuk Gambar?

CNN memanfaatkan struktur gambar.

Gambar memiliki pola lokal seperti:

pixel
 ↓

garis
 ↓

sudut
 ↓

bentuk kecil
 ↓

bentuk yang lebih kompleks
 ↓

objek

Dengan convolution, kernel dapat belajar mendeteksi pola yang sama
di berbagai lokasi gambar.

Karena weights digunakan kembali, jumlah parameter dapat dikurangi
dan kemampuan generalization dapat meningkat.

## Alur CNN

Input Image
     ↓
28 × 28 Image
     ↓
Ambil Subregion 3 × 3
     ↓
Convolution dengan Kernel
     ↓
Kernel Output
     ↓
Tanh
     ↓
Dropout
     ↓
Softmax
     ↓
Prediksi Digit 0–9

## Rangkuman

Chapter 10 memperkenalkan Convolutional Neural Network (CNN).

Konsep utama chapter ini adalah:

1. Reusing Weights
   Weight yang sama digunakan untuk mendeteksi pola yang sama
   pada lokasi berbeda.

2. Kernel
   Kernel adalah kumpulan weights kecil yang digunakan untuk
   mendeteksi pola lokal pada gambar.

3. Convolution
   Kernel diterapkan pada berbagai subregion gambar untuk
   menghasilkan output.

4. Convolutional Layer
   Beberapa kernel dapat digunakan secara bersamaan untuk
   mendeteksi berbagai pola.

5. CNN dan Generalization
   Reusing weights mengurangi jumlah parameter dan membantu
   mengurangi overfitting.

Pada implementasi MNIST, CNN menggunakan kernel 3 × 3 dengan
16 kernel dan menghasilkan sekitar 87.74% test accuracy pada
hasil yang dilaporkan buku.

## Kesimpulan

CNN merupakan neural network yang dirancang untuk memanfaatkan
struktur data gambar.

Ide terpenting dalam chapter ini adalah reusing weights.
Satu kernel dapat mempelajari suatu pola dan kemudian mencari pola
tersebut di berbagai lokasi gambar.

Dengan convolutional layer, neural network dapat belajar mengenali
fitur lokal seperti edge, garis, sudut, dan bentuk kecil.

Implementasi pada MNIST menunjukkan bahwa mengganti layer pertama
dengan convolutional layer dapat meningkatkan performa dibandingkan
network pada chapter sebelumnya.